# Experiments: custom CNN vs ResNet50V2 transfer learning

> **Status: exploratory.** These training runs were not completed, so outputs have been cleared. The finished model and results are in `01_fer2013_cnn.ipynb`.

Uses the image-folder version of FER-2013 (`data/train/<emotion>/`, `data/test/<emotion>/`) with Keras `ImageDataGenerator`, and compares:

1. A custom CNN on 48×48 inputs
2. ResNet50V2 pre-trained on ImageNet, fine-tuned on 224×224 inputs

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('default')

import os
import tensorflow as tf
import keras
import cv2

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import plot_model
from tensorflow.keras import layers, models, optimizers

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, BatchNormalization, MaxPooling2D, Dropout, Flatten, Dense
from tensorflow.keras.applications import ResNet50V2

## 1. Dataset overview

In [ ]:
train_dir = "../data/train/"  # FER-2013 image-folder version, see data/README.md
test_dir = "../data/test/"

def Classes_Count(path, name):
    Classes_Dict = {}
    for Class in os.listdir(path):
        Classes_Dict[Class] = len(os.listdir(os.path.join(path, Class)))
    return pd.DataFrame(Classes_Dict, index=[name])

Train_Count = Classes_Count(train_dir, 'Train').transpose().sort_values(by="Train", ascending=False)
Test_Count = Classes_Count(test_dir, 'Test').transpose().sort_values(by="Test", ascending=False)

In [ ]:
pd.concat([Train_Count,Test_Count], axis=1)

In [ ]:
Train_Count.plot(kind='barh')

In [ ]:
plt.style.use('default')
plt.figure(figsize=(25, 8))
image_count = 1
BASE_URL = train_dir

for directory in os.listdir(BASE_URL):
    if directory[0] != '.':
        dir_path = os.path.join(BASE_URL, directory)
        for i, file in enumerate(os.listdir(dir_path)):
            if i == 1:
                break
            else:
                fig = plt.subplot(1, 7, image_count)
                image_count += 1
                file_path = os.path.join(dir_path, file)
                
                image = cv2.imread(file_path)
                if image is not None:
                    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                    plt.imshow(image_rgb)
                    plt.title(directory, fontsize=20)
                else:
                    print(f"Failed to load image: {file_path}")

plt.tight_layout()
plt.show()

## 2. Custom CNN (48×48)

In [ ]:
img_shape = 48
batch_size = 64
train_data_path = train_dir
test_data_path = test_dir

In [ ]:
# Set up data preprocessing
train_preprocessor = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=10,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

test_preprocessor = ImageDataGenerator(
    rescale=1.0 / 255.0,
)

# Create data generators
train_data = train_preprocessor.flow_from_directory(
    train_data_path,
    class_mode="categorical",
    target_size=(img_shape, img_shape),
    color_mode='rgb',
    shuffle=True,
    batch_size=batch_size
)

test_data = test_preprocessor.flow_from_directory(
    test_data_path,
    class_mode="categorical",
    target_size=(img_shape, img_shape),
    color_mode='rgb',
    shuffle=False,
    batch_size=batch_size
)

In [ ]:
def Create_CNN_Model():

    model = Sequential()

    #CNN1
    model.add(Conv2D(32, (3,3), activation='relu', input_shape=(img_shape,img_shape,3)))
    model.add(BatchNormalization())
    model.add(Conv2D(64, (3,3), activation='relu', padding='same'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size=(2,2), padding='same'))
    model.add(Dropout(0.25))

    #CNN2
    model.add(Conv2D(64, (3,3), activation='relu', ))
    model.add(BatchNormalization())
    model.add(Conv2D(128, (3,3), activation='relu', padding='same'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size=(2,2), padding='same'))
    model.add(Dropout(0.25))

    #CNN3
    model.add(Conv2D(128, (3,3), activation='relu'))
    model.add(BatchNormalization())
    model.add(Conv2D(256, (3,3), activation='relu', padding='same'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size=(2,2), padding='same'))
    model.add(Dropout(0.25))

    #Output

    model.add(Flatten())

    model.add(Dense(1024, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.25))

    model.add(Dense(512, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.25))

    model.add(Dense(256, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.25))

    model.add(Dense(128, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.25))

    model.add(Dense(64, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.25))

    model.add(Dense(32, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.25))

    model.add(Dense(7, activation='softmax'))

    return model


In [ ]:
CNN_Model = Create_CNN_Model()

CNN_Model.summary()

CNN_Model.compile(optimizer="adam", loss='categorical_crossentropy', metrics=['accuracy'])

Callbacks

In [ ]:
checkpoint_path = "CNN_Model_Checkpoint.keras"
Checkpoint = ModelCheckpoint(checkpoint_path, monitor="val_accuracy", save_best_only=True, verbose=1)

Early_Stopping = EarlyStopping(monitor="val_accuracy", patience=15, restore_best_weights=True, verbose=1)

Reducing_LR = tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2, verbose=1)

callbacks = [Checkpoint, Early_Stopping, Reducing_LR]

# Calculate steps per epoch
steps_per_epoch = train_data.n // train_data.batch_size
validation_steps = test_data.n // test_data.batch_size

In [ ]:
CNN_history = CNN_Model.fit(
    train_data,
    validation_data=test_data,
    epochs=25,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
print("Total training samples:", train_data.n)
print("Total validation samples:", test_data.n)
print("Steps per epoch:", steps_per_epoch)
print("Validation steps:", validation_steps)

In [ ]:
CNN_Score = CNN_Model.evaluate(test_data)

print("    Test Loss: {:.5f}".format(CNN_Score[0]))
print("Test Accuracy: {:.2f}".format(CNN_Score[1] * 100))

In [ ]:
def plot_curves(history):
    loss = history.history["loss"]
    val_loss = history.history["val_loss"]

    accuracy = history.history["accuracy"]
    val_accuracy = history.history["val_accuracy"]

    epochs = range(len(history.history["loss"]))

    plt.figure(figsize=(15,5))

    plt.subplot(1,2,1)
    plt.plot(epochs,loss,label = "training_loss")
    plt.plot(epochs,val_loss,label = "val_loss")
    plt.title("Loss")
    plt.xlabel("epochs")
    plt.legend()


    plt.subplot(1,2,2)
    plt.plot(epochs,accuracy,label = "training_accuracy")
    plt.plot(epochs,val_accuracy,label = "val_accuracy")
    plt.title("Accuracy")
    plt.xlabel("epochs")
    plt.legend()
    plt.show()

In [ ]:
plot_curves(CNN_history)

In [ ]:
CNN_Predictions = CNN_Model.predict(test_data)
CNN_Predictions = np.argmax(CNN_Predictions, axis=1)

In [ ]:
test_data.class_indices

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

fig, ax = plt.subplots(figsize=(15,10))

cm = confusion_matrix(test_data.labels, CNN_Predictions)

sns.heatmap(cm, annot=True, fmt='g', ax=ax)

ax.set_xlabel('Predicted labels', fontsize=15, fontweight='bold')
ax.set_ylabel('True labels', fontsize=15, fontweight='bold')
ax.set_title('CNN Confusion Matrix', fontsize=20, fontweight='bold')

## 3. ResNet50V2 transfer learning (224×224)

In [ ]:
img_shape = 224
batch_size = 64
train_data_path = train_dir
test_data_path = test_dir

In [ ]:
# Set up data preprocessing
train_preprocessor = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=10,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

test_preprocessor = ImageDataGenerator(
    rescale=1.0 / 255.0,
)

# Create data generators
train_data = train_preprocessor.flow_from_directory(
    train_data_path,
    class_mode="categorical",
    target_size=(img_shape, img_shape),
    color_mode='rgb',
    shuffle=True,
    batch_size=batch_size
)

test_data = test_preprocessor.flow_from_directory(
    test_data_path,
    class_mode="categorical",
    target_size=(img_shape, img_shape),
    color_mode='rgb',
    shuffle=False,
    batch_size=batch_size
)

In [ ]:
ResNet50V2 = tf.keras.applications.ResNet50V2(input_shape=(224,224,3), include_top = False, weights = 'imagenet')

In [ ]:
ResNet50V2.trainable = True

# Fine-tune only the last 50 layers
for layer in ResNet50V2.layers[:-50]:
    layer.trainable = False

In [ ]:
def Create_ResNet50V2_Model():
    model = Sequential([
        ResNet50V2,
        Dropout(.25),
        BatchNormalization(),
        Flatten(),
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(.5),
        Dense(7, activation='softmax')
    ])
    return model

In [ ]:
ResNet50V2_Model = Create_ResNet50V2_Model()

ResNet50V2_Model.summary()

ResNet50V2_Model.compile(optimizer="adam", loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
checkpoint_path = "ResNet50V2_Model_Checkpoint.keras"

Checkpoint = ModelCheckpoint(checkpoint_path, monitor="val_accuracy", save_best_only=True)

Early_Stopping = EarlyStopping(monitor="val_accuracy", patience=7, restore_best_weights=True, verbose=1)

Reducing_LR = tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2, verbose=1)

callbacks = [Early_Stopping, Reducing_LR]

# Calculate steps per epoch
steps_per_epoch = 28709 // batch_size  # Correctly covers all training samples
validation_steps = 7178 // batch_size  # Correctly covers all validation samples

In [ ]:
ResNet50V2_history = ResNet50V2_Model.fit(
    train_data,
    validation_data=test_data,
    epochs=25,
    batch_size=batch_size,
    callbacks=callbacks,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    verbose=1
)

In [ ]:
ResNet50V2_Score = ResNet50V2_Model.evaluate(test_data)

print("    Test Loss: {:.5f}".format(ResNet50V2_Score[0]))
print("Test Accuracy: {:.2f}".format(ResNet50V2_Score[1] * 100))

In [ ]:
plot_curves(ResNet50V2_history)

In [ ]:
ResNet50V2_Predictions = ResNet50V2_Model.predict(test_data)
ResNet50V2_Predictions = np.argmax(ResNet50V2_Predictions, axis=1)

In [ ]:
fig, ax = plt.subplots(figsize=(15,10))

cm = confusion_matrix(test_data.labels, ResNet50V2_Predictions)

sns.heatmap(cm, annot=True, fmt='g', ax=ax)

ax.set_xlabel('Predicted labels', fontsize=15, fontweight='bold')
ax.set_ylabel('True labels', fontsize=15, fontweight='bold')
ax.set_title('ResNet50V2 Confusion Matrix', fontsize=20, fontweight='bold')

## 4. Sample predictions

In [ ]:
Emotion_Classes = [
    'Angry',
    'Disgust',
    'Fear',
    'Happy',
    'Neutral',
    'Sad',
    'Surprise'
]

In [ ]:
test_preprocessor = ImageDataGenerator(
    rescale=1.0 / 255.0,
)

test_generator = test_preprocessor.flow_from_directory(
    test_data_path,
    class_mode="categorical",
    target_size=(img_shape, img_shape),
    color_mode='rgb',
    shuffle=False,
    batch_size=batch_size
)

In [ ]:
Random_Batch = np.random.randint(0, len(test_generator)- 1) 
Random_Img_Index = np.random.randint(0,batch_size - 1, 10)
fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(25,10), subplot_kw={'xticks':[], 'yticks':[]})

for i,ax in enumerate(axes.flat):
    Random_Img = test_generator[Random_Batch][0][Random_Img_Index[i]]
    Random_Img_Label = np.argmax(test_generator[Random_Batch][1][Random_Img_Index[i]])
    Model_Prediction = np.argmax(ResNet50V2_Model.predict(tf.expand_dims(Random_Img, axis=0), verbose=0))
    ax.imshow(Random_Img)

    if Emotion_Classes[Random_Img_Label] == Emotion_Classes[Model_Prediction]:
        color = "green"
    else:
        color = "red"
    ax.set_title(f"True: {Emotion_Classes[Random_Img_Label]}\nPredicted : {Emotion_Classes[Model_Prediction]}", color=color)
plt.tight_layout()
plt.show()

## 5. Face detection (next step)

In [ ]:
# Haar cascade face detector bundled with OpenCV (first step towards webcam inference)
faceCascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")